In [21]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PATH / "feature_dataset.csv")

print(df.shape)

(90000, 65)


In [22]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,M20,M21,M22,M40,M41,M42,M43,M60,M61,M62,...,SpectralFlatness,SpectralRollOff,SpectralBandwidth,MphiNL,SigmaDP,SigmaZ2,M2M4SNR,Modulation,OriginalSNR,Label
0,0.000026,0.000069,0.000026,3.157343e-09,2.930586e-09,6.460820e-09,2.930586e-09,3.500275e-13,4.633123e-13,4.327737e-13,...,0.627377,43,18.381460,1.0,3.797474,0.005159,1.105342,QPSK,2,8
1,0.000028,0.000068,0.000028,3.215990e-09,2.804059e-09,6.498267e-09,2.804059e-09,1.781424e-13,4.586850e-13,3.564625e-13,...,0.630078,48,19.898510,1.0,2.716261,0.005688,0.889030,QPSK,2,8
2,0.000004,0.000069,0.000004,2.910366e-09,8.069788e-10,7.002352e-09,8.069788e-10,1.588626e-13,3.876018e-13,1.401655e-13,...,0.624587,49,19.985046,1.0,3.927797,0.005326,0.360494,QPSK,2,8
3,0.000005,0.000070,0.000005,3.598735e-09,7.073600e-10,7.253493e-09,7.073600e-10,3.053956e-13,5.095078e-13,1.243595e-13,...,0.606380,47,19.097059,1.0,2.835475,0.006038,0.163557,QPSK,2,8
4,0.000018,0.000068,0.000018,2.224453e-09,2.107830e-09,6.237477e-09,2.107830e-09,1.804345e-13,2.696272e-13,2.822643e-13,...,0.602295,43,19.311969,1.0,2.253482,0.005549,1.231920,QPSK,2,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89995,0.000013,0.000066,0.000013,6.360831e-10,1.167831e-09,5.291343e-09,1.167831e-09,1.077823e-13,4.682848e-14,1.147776e-13,...,0.538621,32,16.009717,1.0,3.014158,0.006250,2.708425,8PSK,6,0
89996,0.000024,0.000068,0.000024,5.657924e-10,2.655918e-09,6.064658e-09,2.655918e-09,9.225986e-14,2.059674e-14,3.234347e-13,...,0.455457,29,15.207054,1.0,1.822241,0.006404,1.712333,8PSK,6,0
89997,0.000009,0.000066,0.000009,5.724184e-10,1.007608e-09,5.252853e-09,1.007608e-09,2.078304e-13,6.945268e-14,1.116262e-13,...,0.457882,33,16.401084,1.0,1.664279,0.006496,2.850204,8PSK,6,0
89998,0.000008,0.000068,0.000008,1.376324e-09,6.879162e-10,5.897059e-09,6.879162e-10,8.783078e-14,1.684706e-13,7.383302e-14,...,0.433737,28,15.480583,1.0,2.498388,0.006668,1.961270,8PSK,6,0


In [23]:
# Fill NaNs only for M2M4SNR
if "M2M4SNR" in df.columns:
    median_value = df["M2M4SNR"].median()
    df["M2M4SNR"] = df["M2M4SNR"].fillna(median_value)

print(df.isnull().sum().sum())
print(df.shape)

0
(90000, 65)


In [24]:
duplicate_columns = [
    "M22",
    "C20",
    "C21",
    "C80",
    "C84",
    "GammaMean",
    "GammaMax"
]

duplicate_columns = [
    col for col in duplicate_columns
    if col in df.columns
]

df.drop(columns=duplicate_columns, inplace=True)

print(df.shape)

(90000, 58)


In [25]:
constant_columns = [
    "WaveletASKCorrelation"
]

constant_columns = [
    col for col in constant_columns
    if col in df.columns
]

df.drop(columns=constant_columns, inplace=True)

print(df.shape)

(90000, 57)


In [26]:
redundant = [
    "M40",
    "M41",
    "M42",
    "M43",
    "M60",
    "M61",
    "M62",
    "M63",
    "M80",
    "M84"
]

redundant = [
    col for col in redundant
    if col in df.columns
]

df.drop(columns=redundant, inplace=True)

print(df.shape)

(90000, 47)


In [27]:
print(df.isnull().sum().sum())

print(np.isinf(df.select_dtypes(np.number)).sum().sum())

print(df.shape)

0
0
(90000, 47)


In [28]:
df.to_csv(
    DATA_PATH / "feature_dataset_clean.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [29]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH / "feature_dataset_clean.csv")

X = df.drop(columns=["Modulation","Label"])

y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(72000, 45)
(18000, 45)


In [30]:
X_train.to_csv(DATA_PATH / "X_train.csv", index=False)
X_test.to_csv(DATA_PATH / "X_test.csv", index=False)

y_train.to_frame("Label").to_csv(DATA_PATH / "y_train.csv", index=False)
y_test.to_frame("Label").to_csv(DATA_PATH / "y_test.csv", index=False)